# 00 · Setup - your sandbox + synthetic data

**Run this first.** It gives you a private schema, a shared warehouse, and a
small synthetic survey dataset that mirrors the KTV clarity/tone survey. Nobody
shares state - your work lives in `WS_<your_user>`.

Run each cell top to bottom. Expected results are shown so you can check yourself.

### 1. Shared database + warehouse
Safe to run repeatedly - first person to run it wins, everyone else no-ops.

In [ ]:
CREATE DATABASE IF NOT EXISTS PLG_CORTEX_WORKSHOP;

CREATE WAREHOUSE IF NOT EXISTS PLG_WORKSHOP_WH
  WAREHOUSE_SIZE = 'MEDIUM'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 3
  COMMENT = 'Shared warehouse for the PLG Cortex workshop';

### 2. Your private sandbox schema
Named after your user so it can't collide with anyone else's.

In [ ]:
SET ws  = REGEXP_REPLACE(CURRENT_USER(), '[^A-Za-z0-9_]', '_');
SET sch = 'PLG_CORTEX_WORKSHOP.WS_' || $ws;
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($sch);

USE WAREHOUSE PLG_WORKSHOP_WH;
USE SCHEMA IDENTIFIER($sch);

SELECT CURRENT_SCHEMA() AS your_sandbox;

### 3. Generate synthetic survey data
`ROWCOUNT => 4000` keeps the whole lab fast. Bump it later (e.g. 20000) if you
want to stress the incremental Dynamic Table in Stage 2.

The data is correlated on purpose: negative Dutch comments + low ratings cluster
with `poor` game rounds, and ~8% of answers are junk (`nee`, `nvt`, `-`) so
`AI_FILTER` has something to remove in Stage 2.

In [ ]:
CREATE OR REPLACE TRANSIENT TABLE _SURVEY_RAW AS
WITH base AS (
  SELECT
    SEQ8()                       AS n,
    UNIFORM(1, 100, RANDOM())    AS mood,
    UNIFORM(1, 1500, RANDOM())   AS player_seed,
    UNIFORM(0, 1, RANDOM())      AS brand_pick,
    UNIFORM(0, 3, RANDOM())      AS email_pick,
    UNIFORM(0, 720, RANDOM())    AS day_offset,
    UNIFORM(0, 4, RANDOM())      AS neg_idx,
    UNIFORM(0, 4, RANDOM())      AS neu_idx,
    UNIFORM(0, 4, RANDOM())      AS pos_idx,
    UNIFORM(0, 3, RANDOM())      AS junk_idx,
    UNIFORM(0, 4, RANDOM())      AS tneg_idx,
    UNIFORM(0, 4, RANDOM())      AS tpos_idx,
    UNIFORM(1, 100, RANDOM())    AS junk_roll
  FROM TABLE(GENERATOR(ROWCOUNT => 4000))
)
SELECT
  'R'  || LPAD(n::string, 7, '0')                                AS response_id,
  'P'  || LPAD(player_seed::string, 5, '0')                      AS player_id,
  'GR' || LPAD(n::string, 7, '0')                                AS game_round_id,
  CASE WHEN brand_pick = 0 THEN 'NPL' ELSE 'VriendenLoterij' END AS brand,
  GET(ARRAY_CONSTRUCT('welcome','prize_notification','monthly_update','winback'), email_pick)::string AS email_type,
  DATEADD('day', -day_offset, CURRENT_DATE())                    AS survey_date,
  CASE WHEN mood <= 30 THEN 'poor' WHEN mood <= 70 THEN 'normal' ELSE 'good' END AS game_round_performance,
  ROUND(CASE WHEN mood <= 30 THEN UNIFORM(0.0, 0.3, RANDOM())
             WHEN mood <= 70 THEN UNIFORM(0.2, 0.6, RANDOM())
             ELSE UNIFORM(0.5, 0.95, RANDOM()) END, 2)           AS replay_rate,
  CASE WHEN mood <= 30 THEN UNIFORM(1,3,RANDOM())
       WHEN mood <= 70 THEN UNIFORM(2,4,RANDOM())
       ELSE UNIFORM(4,5,RANDOM()) END                            AS clarity_rating,
  CASE WHEN mood <= 30 THEN UNIFORM(1,3,RANDOM())
       WHEN mood <= 70 THEN UNIFORM(2,4,RANDOM())
       ELSE UNIFORM(3,5,RANDOM()) END                            AS tone_rating,
  CASE
    WHEN junk_roll <= 8 THEN GET(ARRAY_CONSTRUCT('nee','nvt','geen','-'), junk_idx)::string
    WHEN mood <= 30 THEN GET(ARRAY_CONSTRUCT(
        'De e-mail was verwarrend, ik snapte niet wat ik moest doen.',
        'Te veel tekst en onduidelijke uitleg over de trekking.',
        'Ik begreep de actievoorwaarden totaal niet.',
        'Onduidelijk welke prijs ik had gewonnen.',
        'De knop werkte niet en de instructies klopten niet.'), neg_idx)::string
    WHEN mood <= 70 THEN GET(ARRAY_CONSTRUCT(
        'Redelijk duidelijk, maar kan korter.',
        'Grotendeels helder, een paar zinnen waren wat lang.',
        'Prima, al miste ik wat details over de einddatum.',
        'Op zich oke, de layout mag rustiger.',
        'Voldoende duidelijk voor mij.'), neu_idx)::string
    ELSE GET(ARRAY_CONSTRUCT(
        'Heel duidelijke e-mail, precies wat ik nodig had.',
        'Fijn en helder geschreven, top.',
        'Duidelijke uitleg over mijn prijs, dank!',
        'Prettige toon en goed leesbaar.',
        'Alles was meteen duidelijk.'), pos_idx)::string
  END                                                            AS clarity_comment,
  CASE
    WHEN junk_roll BETWEEN 90 AND 94 THEN GET(ARRAY_CONSTRUCT('nvt','nee','-','geen mening'), junk_idx)::string
    WHEN mood <= 30 THEN GET(ARRAY_CONSTRUCT(
        'De toon voelde afstandelijk en onpersoonlijk.',
        'Kwam nogal pusherig over.',
        'Te commercieel, niet echt vriendelijk.',
        'Ik voelde me niet serieus genomen.',
        'De toon was koel en zakelijk.'), tneg_idx)::string
    WHEN mood <= 70 THEN GET(ARRAY_CONSTRUCT(
        'Neutrale toon, prima.',
        'Vriendelijk genoeg.',
        'Zakelijk maar oke.',
        'Niet storend, niet bijzonder.',
        'Redelijk warme toon.'), neu_idx)::string
    ELSE GET(ARRAY_CONSTRUCT(
        'Warme, persoonlijke toon. Voelde vriendelijk.',
        'Heel prettig en respectvol geschreven.',
        'Enthousiaste en positieve toon, leuk!',
        'Voelde persoonlijk en betrokken.',
        'Fijne, gastvrije toon.'), tpos_idx)::string
  END                                                            AS tone_comment
FROM base;

### 4. Split into the three tables the workshop joins

In [ ]:
CREATE OR REPLACE TABLE SURVEY_RESPONSES AS
  SELECT response_id, player_id, game_round_id, brand, email_type, survey_date,
         clarity_rating, tone_rating, clarity_comment, tone_comment
  FROM _SURVEY_RAW;

CREATE OR REPLACE TABLE GAME_ROUNDS AS
  SELECT game_round_id, player_id, survey_date AS round_date, game_round_performance
  FROM _SURVEY_RAW;

CREATE OR REPLACE TABLE PLAYER_BEHAVIOUR AS
  SELECT player_id, AVG(replay_rate) AS replay_rate, COUNT(*) AS surveys_answered
  FROM _SURVEY_RAW GROUP BY player_id;

DROP TABLE IF EXISTS _SURVEY_RAW;

### 5. Self-check 
Expect ~4000 survey rows, ~4000 game rounds, and up to 1500 players.

In [ ]:
SELECT 'SURVEY_RESPONSES' AS tbl, COUNT(*) AS row_count FROM SURVEY_RESPONSES
UNION ALL SELECT 'GAME_ROUNDS', COUNT(*) FROM GAME_ROUNDS
UNION ALL SELECT 'PLAYER_BEHAVIOUR', COUNT(*) FROM PLAYER_BEHAVIOUR;

### Region check (if AI functions error later)
If Stage 2 says a Cortex function isn't available in your region, an admin can run:
```sql
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';
```

**Done - move on to `01_foundation_semantic_view.ipynb`.**